# MSigDB decoupleR ORA Coverage Plots: CLAMPfull and CLAMPbase (by study subsampling)

**Environment:** `clamp-analyses`

Reads RDS files produced by `00_msigdb_decoupler_ora_analysis.ipynb` and plots MSigDB pathway coverage (FDR < 0.05 / FDR < 0.01) against sample proportion. Discovers available percentages dynamically: no hardcoded values. Plots are displayed inline only — no PDF export.

In [ ]:
library(here)
library(ggplot2)
library(dplyr)

input_dir  <- here("output/03_model_biology/00_archs4/06_coverage_study/decoupler_ora")

theme_ng <- function() {
  theme_classic(base_size = 18) +
    theme(
      axis.title        = element_text(size = 24, colour = "black"),
      axis.title.y      = element_text(margin = margin(r = 18)),
      axis.text         = element_text(size = 18, colour = "black"),
      axis.line         = element_line(linewidth = 0.8, colour = "black"),
      axis.ticks        = element_line(linewidth = 0.8, colour = "black"),
      axis.ticks.length = unit(0.22, "cm"),
      panel.grid.major.y = element_line(colour = "#BDBDBD", linewidth = 0.7),
      panel.grid.minor   = element_blank(),
      legend.position    = "none",
      plot.title         = element_blank(),
      strip.background   = element_blank(),
      strip.text         = element_text(face = "bold", size = 18),
      plot.margin        = margin(t = 16, r = 12, b = 12, l = 42)
    )
}

## Load results

In [ ]:
load_model_results <- function(model_subdir) {
  rds_files <- list.files(
    file.path(input_dir, model_subdir),
    pattern    = "^rs[0-9]+_seed[0-9]+_msigdb_decoupler_ora\\.rds$",
    full.names = TRUE
  )
  if (length(rds_files) == 0) {
    warning(sprintf("No per-model _msigdb_decoupler_ora.rds files found for %s", model_subdir))
    return(NULL)
  }
  message(sprintf("Loading %d per-model caches from %s", length(rds_files), model_subdir))

  rows <- lapply(rds_files, function(f) {
    m <- regmatches(basename(f), regexec("^rs([0-9]+)_seed([0-9]+)_msigdb_decoupler_ora\\.rds$", basename(f)))[[1]]
    if (length(m) < 3) return(NULL)
    res <- readRDS(f)
    if (is.null(res$terms_padj)) return(NULL)
    data.frame(
      coverage_pct       = as.integer(m[2]),
      seed               = as.integer(m[3]),
      n_samples          = res$n_samples,
      n_lvs              = res$n_lvs,
      n_total_msigdb     = res$n_total_msigdb,
      coverage_msigdb_05 = sum(res$terms_padj < 0.05) / res$n_total_msigdb,
      coverage_msigdb_01 = sum(res$terms_padj < 0.01) / res$n_total_msigdb,
      stringsAsFactors   = FALSE
    )
  })
  df <- do.call(rbind, Filter(Negate(is.null), rows))
  df[order(df$coverage_pct, df$seed), ]
}

df_full <- load_model_results("CLAMPfull")
df_base <- load_model_results("CLAMPbase")

ref_df <- if (!is.null(df_full)) df_full else df_base

sample_counts <- ref_df %>%
  dplyr::group_by(coverage_pct) %>%
  dplyr::summarise(med_n = round(median(n_samples, na.rm = TRUE)), .groups = "drop") %>%
  dplyr::arrange(coverage_pct)

label_map <- setNames(
  paste0(format(sample_counts$med_n, big.mark = ","),
         " samples (", sample_counts$coverage_pct, "%)"),
  sample_counts$coverage_pct
)

add_labels <- function(df) {
  if (is.null(df)) return(NULL)
  df %>%
    dplyr::mutate(
      coverage_label         = factor(label_map[as.character(coverage_pct)], levels = label_map),
      coverage_msigdb_05_pct = coverage_msigdb_05 * 100,
      coverage_msigdb_01_pct = coverage_msigdb_01 * 100
    )
}

df_full <- add_labels(df_full)
df_base <- add_labels(df_base)

message("CLAMPfull: ", nrow(df_full), " rows | coverage levels: ",
        paste(sort(unique(df_full$coverage_pct)), collapse = ", "), "%")
message("CLAMPbase: ", nrow(df_base), " rows | coverage levels: ",
        paste(sort(unique(df_base$coverage_pct)), collapse = ", "), "%")

## Plot helper

In [ ]:
make_coverage_plot <- function(df, y_col, y_label, label_offset = 2) {
  if (is.null(df)) { message("No data, skipping plot"); return(invisible(NULL)) }
  purple_fill <- "#9b59b6"

  label_df <- df %>%
    dplyr::group_by(coverage_label) %>%
    dplyr::summarise(
      mean_y  = mean(.data[[y_col]], na.rm = TRUE),
      max_y   = max(.data[[y_col]],  na.rm = TRUE),
      label_y = max_y + label_offset,
      label   = sprintf("%.1f%%", mean_y),
      .groups = "drop"
    )

  ggplot(df, aes(x = coverage_label, y = .data[[y_col]])) +
    geom_boxplot(
      width = 0.55, fill = scales::alpha(purple_fill, 0.80),
      colour = "black", linewidth = 0.8, outlier.shape = NA
    ) +
    geom_jitter(
      width = 0.08, size = 2.8, shape = 21,
      fill = "white", colour = "#2b2b2b", stroke = 0.5, alpha = 0.95
    ) +
    stat_summary(fun = mean, geom = "point", shape = 23, size = 3.8,
                 fill = "white", colour = "black", stroke = 0.8) +
    stat_summary(fun = mean, geom = "line", aes(group = 1),
                 colour = purple_fill, linewidth = 1.0, linetype = "dashed") +
    geom_text(data = label_df,
              aes(x = coverage_label, y = label_y, label = label),
              inherit.aes = FALSE, size = 5.5, fontface = "bold", vjust = 0) +
    scale_y_continuous(limits = c(0, NA), expand = expansion(mult = c(0, 0.22))) +
    labs(x = "Proportion of samples", y = y_label) +
    coord_cartesian(clip = "off") +
    theme_ng() +
    theme(axis.text.x = element_text(angle = 30, hjust = 1, vjust = 1))
}

## CLAMPfull

In [ ]:
options(repr.plot.width = 10, repr.plot.height = 10)
p_full_05 <- make_coverage_plot(df_full, "coverage_msigdb_05_pct",
                                "MSigDB pathway coverage, FDR < 0.05 (%)")
p_full_05

In [ ]:
p_full_01 <- make_coverage_plot(df_full, "coverage_msigdb_01_pct",
                                "MSigDB pathway coverage, FDR < 0.01 (%)")
p_full_01

## CLAMPbase

In [ ]:
p_base_05 <- make_coverage_plot(df_base, "coverage_msigdb_05_pct",
                                "MSigDB pathway coverage, FDR < 0.05 (%)")
p_base_05

In [ ]:
p_base_01 <- make_coverage_plot(df_base, "coverage_msigdb_01_pct",
                                "MSigDB pathway coverage, FDR < 0.01 (%)")
p_base_01